# Gold Layer — Metadata Compliance Engine

**Catalog:** metadata_governance  
**Schema:** gold  
**Tables:** gold_metadata, gold_certified_metadata, gold_quality_metrics  
**Layer:** Gold (certified, governed, compliance-scored metadata)  
**Source:** metadata_governance.silver.silver_metadata + metadata_governance.silver.bronze_metadata_quarantine  
**Governance Rules:** R01–R09 (Completeness, Ownership, Security, Structure, Privacy, Business Glossary)  
**Purpose:** Promote clean Silver metadata to Gold, apply 9 governance rules, assign PASS/FAIL certification status, calculate quality scores, and write certified results to Gold layer for Power BI dashboards and Genie AI.


## Step 0: Promote Silver to Gold Base Table

Write clean Silver metadata to the Gold layer as a base table. This matches the architecture where the Compliance Engine reads from Gold — not directly from Silver.

In [0]:
from pyspark.sql.functions import (
    col, when, lit, concat_ws, length, trim,
    round, avg, count, sum as spark_sum, current_timestamp
)

# Read from Silver — Clean validated output
silver_df = spark.table("metadata_governance.silver.silver_metadata")
silver_count = silver_df.count()
print(f"Silver rows loaded: {silver_count}")

# Promote to Gold base table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.gold.gold_metadata")

gold_check = spark.table("metadata_governance.gold.gold_metadata").count()
print("=" * 55)
print("  STEP 0 — Silver Promoted to Gold")
print("=" * 55)
print(f"  Silver rows read  : {silver_count}")
print(f"  Gold rows written : {gold_check}")
print(f"  metadata_governance.gold.gold_metadata ")
print("=" * 55)

## Step 1: Read Metadata from Gold Layer

Read the promoted metadata from the Gold layer. This is the official input to the Compliance Engine, matching the architecture diagram where the Compliance Engine sits to the right of Gold.

In [0]:
# Read from Gold — matches architecture diagram
gold_df = spark.table("metadata_governance.gold.gold_metadata")
gold_count = gold_df.count()

print(f"Gold rows loaded: {gold_count}")
print(f"Columns available: {len(gold_df.columns)}")
display(gold_df.limit(5))

## Step 2: Read Failed Records from Quarantine

Pipeline sent 457 invalid records to the quarantine table via the Alert path. The Compliance Engine must also process these records and flag them with their ingestion failure reason plus governance verdict on top.

In [0]:
# Read quarantine — Structurally failed records
quarantine_df = spark.table("metadata_governance.silver.bronze_metadata_quarantine")
quarantine_count = quarantine_df.count()

print("=" * 55)
print("  STEP 2 — Quarantine Records Loaded")
print("=" * 55)
print(f"  Gold rows          : {gold_count}")
print(f"  Quarantine rows    : {quarantine_count}")
print(f"  Total to process   : {gold_count + quarantine_count}")
print("=" * 55)
display(quarantine_df.limit(5))

## Step 3: Define Governance Rules R01-R09

Define the 9 governance rules every metadata record must satisfy. These rules are applied to both Gold records and Quarantine records.


In [0]:
# Valid values
valid_security  = ["public", "internal", "confidential", "restricted"]
valid_schemas   = ["bronze", "silver", "gold"]
pii_restricted  = ["confidential", "restricted"]

# R01 — Description checks
r01_col_desc   = col("column_desc").isNotNull() & (length(trim(col("column_desc"))) >= 10)
r01_table_desc = col("table_desc").isNotNull() & (length(trim(col("table_desc"))) >= 10)

# R02 — Data steward check
r02_steward = col("data_steward").isNotNull() & (trim(col("data_steward")) != "")

# R03 — Security classification check
r03_security = col("security_classification").isin(valid_security)

# R04 — Schema name check (hard override)
r04_schema = col("schema_name").isin(valid_schemas)

# R05 — Table must have rows
r05_row_count = col("total_record_count").isNotNull() & (col("total_record_count") > 0)

# R06 — PII flag must be set
r06_pii = col("pii_flag").isNotNull()

# R07 — Critical data element flag must be set
r07_cde = col("critical_data_element_flag").isNotNull()

# R08 — Business term must be assigned
r08_term = col("term_name").isNotNull() & (trim(col("term_name")) != "")

# R09 — Cross-field: if PII = true then security must be confidential or restricted
r09_pii_security = (
    (col("pii_flag") != True) |
    (col("security_classification").isin(pii_restricted))
)

print("=" * 55)
print("  STEP 3 — Governance Rules Defined (R01–R09)")
print("=" * 55)
print(f"  R01 → column_desc: not null, min 10 chars")
print(f"  R01 → table_desc: not null, min 10 chars")
print(f"  R02 → data_steward: not null or blank")
print(f"  R03 → security_classification in {valid_security}")
print(f"  R04 → schema_name in {valid_schemas} (hard override)")
print(f"  R05 → total_record_count: not null and > 0")
print(f"  R06 → pii_flag: not null")
print(f"  R07 → critical_data_element_flag: not null")
print(f"  R08 → term_name: not null or blank")
print(f"  R09 → pii=true requires confidential or restricted classification")
print("=" * 55)

## Step 4: Apply Governance Rules to Gold Records

Apply R01–R09 to every Gold record. Assign certification_status, error_detail, remediation_hint, completeness_score and ingestion_status.


In [0]:
def apply_governance_rules(df, ingestion_status_value):
    return df \
        .withColumn(
            "error_detail",
            concat_ws(" | ",
                when(~r01_col_desc,     lit("R01: column_desc missing or too short (min 10 chars)")),
                when(~r01_table_desc,   lit("R01: table_desc missing or too short (min 10 chars)")),
                when(~r02_steward,      lit("R02: data_steward missing")),
                when(~r03_security,     lit("R03: security_classification missing or invalid")),
                when(~r04_schema,       lit("R04: schema_name not recognised")),
                when(~r05_row_count,    lit("R05: table has no rows or row count is null")),
                when(~r06_pii,          lit("R06: pii_flag is not set")),
                when(~r07_cde,          lit("R07: critical_data_element_flag is not set")),
                when(~r08_term,         lit("R08: no business term assigned")),
                when(~r09_pii_security, lit("R09: PII column must have confidential or restricted classification"))
            )
        ) \
        .withColumn(
            "remediation_hint",
            concat_ws(" | ",
                when(~r01_col_desc,     lit("Add a column description of at least 10 characters")),
                when(~r01_table_desc,   lit("Add a table description of at least 10 characters")),
                when(~r02_steward,      lit("Assign a responsible data steward or team")),
                when(~r03_security,     lit("Set security_classification to: public, internal, confidential or restricted")),
                when(~r04_schema,       lit("Move table to correct schema: bronze, silver or gold")),
                when(~r05_row_count,    lit("Check if this table has data — empty tables should be investigated")),
                when(~r06_pii,          lit("Set pii_flag to true or false for this column")),
                when(~r07_cde,          lit("Set critical_data_element_flag to true or false for this column")),
                when(~r08_term,         lit("Assign a business glossary term to this column")),
                when(~r09_pii_security, lit("PII data must be classified as confidential or restricted"))
            )
        ) \
        .withColumn(
            "certification_status",
            when(
                r01_col_desc & r01_table_desc & r02_steward & r03_security &
                r04_schema & r05_row_count & r06_pii & r07_cde & r08_term & r09_pii_security,
                lit("PASS")
            ).otherwise(lit("FAIL"))
        ) \
        .withColumn("r01_col_pass",   when(r01_col_desc,     lit(1)).otherwise(lit(0))) \
        .withColumn("r01_table_pass", when(r01_table_desc,   lit(1)).otherwise(lit(0))) \
        .withColumn("r02_pass",       when(r02_steward,      lit(1)).otherwise(lit(0))) \
        .withColumn("r03_pass",       when(r03_security,     lit(1)).otherwise(lit(0))) \
        .withColumn("r05_pass",       when(r05_row_count,    lit(1)).otherwise(lit(0))) \
        .withColumn("r06_pass",       when(r06_pii,          lit(1)).otherwise(lit(0))) \
        .withColumn("r07_pass",       when(r07_cde,          lit(1)).otherwise(lit(0))) \
        .withColumn("r08_pass",       when(r08_term,         lit(1)).otherwise(lit(0))) \
        .withColumn("r09_pass",       when(r09_pii_security, lit(1)).otherwise(lit(0))) \
        .withColumn(
            "completeness_score",
            (
                col("r01_col_pass") + col("r01_table_pass") +
                col("r02_pass") + col("r03_pass") +
                col("r05_pass") + col("r06_pass") +
                col("r07_pass") + col("r08_pass") +
                col("r09_pass")
            ) * lit(100) / lit(9)
        ) \
        .withColumn("completeness_score", col("completeness_score").cast("string")) \
        .drop(
            "r01_col_pass", "r01_table_pass", "r02_pass", "r03_pass",
            "r05_pass", "r06_pass", "r07_pass", "r08_pass", "r09_pass"
        ) \
        .withColumn(
            "error_detail",
            when(col("certification_status") == "PASS", lit(None)).otherwise(col("error_detail"))
        ) \
        .withColumn(
            "remediation_hint",
            when(col("certification_status") == "PASS", lit(None)).otherwise(col("remediation_hint"))
        ) \
        .withColumn("ingestion_status", lit(ingestion_status_value))

# Apply to Gold records
df_gold_certified = apply_governance_rules(gold_df, "PASSED_SILVER")

pass_gold = df_gold_certified.filter(col("certification_status") == "PASS").count()
fail_gold = df_gold_certified.filter(col("certification_status") == "FAIL").count()
pass_rate = int(pass_gold) / int(gold_count) * 100

print("=" * 55)
print("  STEP 4 — Gold Records Certification Results")
print("=" * 55)
print(f"  Total records : {gold_count}")
print(f"  PASS          : {pass_gold}")
print(f"  FAIL          : {fail_gold}")
print(f"  Pass rate     : {pass_rate:.2f}%")
print("=" * 55)

df_gold_certified.select(
    "table_name",
    "column_name",
    "certification_status",
    "error_detail",
    "ingestion_status"
).show(10, truncate=False)

## Step 5: Apply Governance Rules to Quarantine Records

Apply R01-R09 to failed records. These records automatically receive FAIL certification and are flagged with both ingestion failure reason and any additional governance failures.

In [0]:
# Apply governance rules to quarantine records
df_quarantine_certified = apply_governance_rules(quarantine_df, "FAILED_SILVER")

# Override certification_status to FAIL
df_quarantine_certified = df_quarantine_certified.withColumn(
    "certification_status", lit("FAIL")
).withColumn(
    "error_detail",
    concat_ws(" | ",
        concat_ws(": ", lit("INGESTION FAIL"), col("failure_reason")),
        col("error_detail")
    )
)

# Drop failure_reason before union
df_quarantine_final = df_quarantine_certified.drop("failure_reason")

print("=" * 55)
print("  STEP 5 — Quarantine Records Certification Results")
print("=" * 55)
print(f"  Total quarantine records : {quarantine_count}")
print(f"  All records              : FAIL (failed ingestion)")
print("=" * 55)

df_quarantine_final.select(
    "table_name",
    "column_name",
    "certification_status",
    "error_detail",
    "ingestion_status"
).show(10, truncate=False)

## Step 6: Combine Gold + Quarantine into Complete Dataset

Union Gold certified records and Quarantine certified records into one complete dataset covering all 10,000 original records.


In [0]:
# Union Gold + Quarantine
df_full = df_gold_certified.unionByName(
    df_quarantine_final,
    allowMissingColumns=True
)

total_full = df_full.count()
pass_full  = df_full.filter(col("certification_status") == "PASS").count()
fail_full  = df_full.filter(col("certification_status") == "FAIL").count()
overall_rate = int(pass_full) / int(total_full) * 100

print("=" * 55)
print("  STEP 6 — COMPLETE COMPLIANCE PICTURE")
print("=" * 55)
print(f"  Gold PASS                : {pass_gold}")
print(f"  Gold FAIL                : {fail_gold}")
print(f"  Quarantine FAIL          : {quarantine_count}")
print(f"  Total records processed  : {total_full}")
print(f"  Overall PASS             : {pass_full}")
print(f"  Overall FAIL             : {fail_full}")
print(f"  Overall compliance score : {overall_rate:.2f}%")
print("=" * 55)

# Step 7: Calculate Quality Metrics

Calculate completeness scores at 4 levels: field level, table level, schema level, database level, and overall catalog compliance score.

In [0]:
# ── FIELD LEVEL METRICS ──────────────────────────────
field_metrics = df_full.agg(
    avg(when(r01_col_desc,     lit(100)).otherwise(lit(0))).alias("R01_column_desc_%"),
    avg(when(r01_table_desc,   lit(100)).otherwise(lit(0))).alias("R01_table_desc_%"),
    avg(when(r02_steward,      lit(100)).otherwise(lit(0))).alias("R02_data_steward_%"),
    avg(when(r03_security,     lit(100)).otherwise(lit(0))).alias("R03_security_classification_%"),
    avg(when(r04_schema,       lit(100)).otherwise(lit(0))).alias("R04_schema_name_%"),
    avg(when(r05_row_count,    lit(100)).otherwise(lit(0))).alias("R05_row_count_%"),
    avg(when(r06_pii,          lit(100)).otherwise(lit(0))).alias("R06_pii_flag_%"),
    avg(when(r07_cde,          lit(100)).otherwise(lit(0))).alias("R07_critical_flag_%"),
    avg(when(r08_term,         lit(100)).otherwise(lit(0))).alias("R08_term_name_%"),
    avg(when(r09_pii_security, lit(100)).otherwise(lit(0))).alias("R09_pii_security_%")
)

print("=" * 55)
print("  FIELD LEVEL METRICS — All 9 Rules")
print("=" * 55)
field_row = field_metrics.collect()[0]
print(f"  R01 column_desc             : {field_row['R01_column_desc_%']:.2f}%")
print(f"  R01 table_desc              : {field_row['R01_table_desc_%']:.2f}%")
print(f"  R02 data_steward            : {field_row['R02_data_steward_%']:.2f}%")
print(f"  R03 security_classification : {field_row['R03_security_classification_%']:.2f}%")
print(f"  R04 schema_name             : {field_row['R04_schema_name_%']:.2f}%")
print(f"  R05 row_count               : {field_row['R05_row_count_%']:.2f}%")
print(f"  R06 pii_flag                : {field_row['R06_pii_flag_%']:.2f}%")
print(f"  R07 critical_flag           : {field_row['R07_critical_flag_%']:.2f}%")
print(f"  R08 term_name               : {field_row['R08_term_name_%']:.2f}%")
print(f"  R09 pii_security            : {field_row['R09_pii_security_%']:.2f}%")
print("=" * 55)

# ── RULE BREAKDOWN — Failure count per rule ───────────
rule_breakdown = df_full.agg(
    spark_sum(when(~r01_col_desc,     lit(1)).otherwise(lit(0))).alias("R01_col_desc_failures"),
    spark_sum(when(~r01_table_desc,   lit(1)).otherwise(lit(0))).alias("R01_table_desc_failures"),
    spark_sum(when(~r02_steward,      lit(1)).otherwise(lit(0))).alias("R02_steward_failures"),
    spark_sum(when(~r03_security,     lit(1)).otherwise(lit(0))).alias("R03_security_failures"),
    spark_sum(when(~r04_schema,       lit(1)).otherwise(lit(0))).alias("R04_schema_failures"),
    spark_sum(when(~r05_row_count,    lit(1)).otherwise(lit(0))).alias("R05_row_count_failures"),
    spark_sum(when(~r06_pii,          lit(1)).otherwise(lit(0))).alias("R06_pii_failures"),
    spark_sum(when(~r07_cde,          lit(1)).otherwise(lit(0))).alias("R07_cde_failures"),
    spark_sum(when(~r08_term,         lit(1)).otherwise(lit(0))).alias("R08_term_failures"),
    spark_sum(when(~r09_pii_security, lit(1)).otherwise(lit(0))).alias("R09_pii_security_failures")
)

print("=" * 55)
print("  RULE BREAKDOWN — Failure Count Per Rule")
print("=" * 55)
bd_row = rule_breakdown.collect()[0]
print(f"  R01 column_desc failures      : {bd_row['R01_col_desc_failures']}")
print(f"  R01 table_desc failures       : {bd_row['R01_table_desc_failures']}")
print(f"  R02 data_steward failures     : {bd_row['R02_steward_failures']}")
print(f"  R03 security failures         : {bd_row['R03_security_failures']}")
print(f"  R04 schema failures           : {bd_row['R04_schema_failures']}")
print(f"  R05 row count failures        : {bd_row['R05_row_count_failures']}")
print(f"  R06 pii_flag failures         : {bd_row['R06_pii_failures']}")
print(f"  R07 critical flag failures    : {bd_row['R07_cde_failures']}")
print(f"  R08 term_name failures        : {bd_row['R08_term_failures']}")
print(f"  R09 pii_security failures     : {bd_row['R09_pii_security_failures']}")
print("=" * 55)

# ── TABLE LEVEL METRICS ──────────────────────────────
table_metrics = df_full.groupBy("table_name").agg(
    count("*").alias("total_columns"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_columns"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("table_compliance_%")
).orderBy("table_compliance_%")

print("=" * 55)
print("  TABLE LEVEL METRICS")
print("=" * 55)
for row in table_metrics.collect():
    tbl_name   = row["table_name"]
    tbl_score  = row["table_compliance_%"]
    tbl_passed = row["passed_columns"]
    tbl_total  = row["total_columns"]
    print(f"  {tbl_name}: {tbl_score:.2f}% ({tbl_passed}/{tbl_total} cols)")
print("=" * 55)

# ── SCHEMA LEVEL METRICS ─────────────────────────────
schema_metrics = df_full.groupBy("schema_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("schema_compliance_%")
).orderBy("schema_compliance_%")

print("=" * 55)
print("  SCHEMA LEVEL METRICS")
print("=" * 55)
for row in schema_metrics.collect():
    sch_name   = row["schema_name"]
    sch_score  = row["schema_compliance_%"]
    sch_passed = row["passed_records"]
    sch_total  = row["total_records"]
    print(f"  {sch_name}: {sch_score:.2f}% ({sch_passed}/{sch_total} records)")
print("=" * 55)

# ── DATABASE LEVEL METRICS ───────────────────────────
db_metrics = df_full.groupBy("database_name").agg(
    count("*").alias("total_records"),
    spark_sum(when(col("certification_status") == "PASS", 1).otherwise(0)).alias("passed_records"),
    avg(when(col("certification_status") == "PASS", lit(100)).otherwise(lit(0))).alias("database_compliance_%")
).orderBy("database_compliance_%")

print("=" * 55)
print("  DATABASE LEVEL METRICS")
print("=" * 55)
for row in db_metrics.collect():
    db_name   = row["database_name"]
    db_score  = row["database_compliance_%"]
    db_passed = row["passed_records"]
    db_total  = row["total_records"]
    print(f"  {db_name}: {db_score:.2f}% ({db_passed}/{db_total} records)")
print("=" * 55)

# ── OVERALL CATALOG SCORE ────────────────────────────
overall_score = overall_rate
print("=" * 55)
print("  OVERALL CATALOG COMPLIANCE")
print("=" * 55)
print(f"  Overall compliance score : {overall_score:.2f}%")
print(f"  Total certified PASS     : {pass_full}")
print(f"  Total FAIL               : {fail_full}")
print(f"  Total records            : {total_full}")
print("=" * 55)

# Step 8: Write Certified Output to Gold 

Write two tables to the Gold layer — gold_certified_metadata and gold_quality_metrics — both with snapshot timestamps for the Metadata HUB history tracking.


In [0]:
from pyspark.sql import Row

# Add snapshot timestamp
df_final = df_full.withColumn("certified_at", current_timestamp())

# ── DROP AND RECREATE TABLE TO AVOID SCHEMA CONFLICTS ─
spark.sql("DROP TABLE IF EXISTS metadata_governance.gold.gold_certified_metadata")

# ── WRITE CERTIFIED METADATA TABLE ───────────────────
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("metadata_governance.gold.gold_certified_metadata")

certified_count = spark.table("metadata_governance.gold.gold_certified_metadata").count()
print(f"gold_certified_metadata written: {certified_count} rows")

# ── BUILD QUALITY METRICS TABLE ───────────────────────
metrics_rows = []

# Catalog level
metrics_rows.append(Row(
    level="catalog",
    name="metadata_governance",
    total_records=total_full,
    passed=pass_full,
    failed=fail_full,
    compliance_score=float(overall_score)
))

# Schema level
for row in schema_metrics.collect():
    metrics_rows.append(Row(
        level="schema",
        name=str(row["schema_name"]),
        total_records=int(row["total_records"]),
        passed=int(row["passed_records"]),
        failed=int(row["total_records"]) - int(row["passed_records"]),
        compliance_score=float(row["schema_compliance_%"])
    ))

# Table level
for row in table_metrics.collect():
    metrics_rows.append(Row(
        level="table",
        name=str(row["table_name"]),
        total_records=int(row["total_columns"]),
        passed=int(row["passed_columns"]),
        failed=int(row["total_columns"]) - int(row["passed_columns"]),
        compliance_score=float(row["table_compliance_%"])
    ))

# Database level
for row in db_metrics.collect():
    metrics_rows.append(Row(
        level="database",
        name=str(row["database_name"]),
        total_records=int(row["total_records"]),
        passed=int(row["passed_records"]),
        failed=int(row["total_records"]) - int(row["passed_records"]),
        compliance_score=float(row["database_compliance_%"])
    ))

# Rule breakdown level (NEW)
bd_row = rule_breakdown.collect()[0]
rule_names = [
    ("R01_column_desc",          bd_row["R01_col_desc_failures"]),
    ("R01_table_desc",           bd_row["R01_table_desc_failures"]),
    ("R02_data_steward",         bd_row["R02_steward_failures"]),
    ("R03_security_classification", bd_row["R03_security_failures"]),
    ("R04_schema_name",          bd_row["R04_schema_failures"]),
    ("R05_row_count",            bd_row["R05_row_count_failures"]),
    ("R06_pii_flag",             bd_row["R06_pii_failures"]),
    ("R07_critical_flag",        bd_row["R07_cde_failures"]),
    ("R08_term_name",            bd_row["R08_term_failures"]),
    ("R09_pii_security",         bd_row["R09_pii_security_failures"]),
]
for rule_name, failures in rule_names:
    metrics_rows.append(Row(
        level="rule",
        name=rule_name,
        total_records=total_full,
        passed=total_full - int(failures),
        failed=int(failures),
        compliance_score=float((total_full - int(failures)) / total_full * 100)
    ))

# Write metrics table
df_metrics = spark.createDataFrame(metrics_rows) \
    .withColumn("calculated_at", current_timestamp())

df_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("metadata_governance.gold.gold_quality_metrics")

metrics_count = spark.table("metadata_governance.gold.gold_quality_metrics").count()

print("=" * 55)
print("  STEP 8 — Gold Layer Write Complete")
print("=" * 55)
print(f"  gold_metadata           : {gold_count} rows (base table)")
print(f"  gold_certified_metadata : {certified_count} rows")
print(f"  gold_quality_metrics    : {metrics_count} rows")
print("=" * 55)

# Step 9: Final Verification

Confirm all Gold tables were written correctly and display sample data for the demo.

In [0]:
# Verify gold_metadata (base table)
gold_base = spark.table("metadata_governance.gold.gold_metadata")
print(f"gold_metadata (base): {gold_base.count()} rows")

# Verify gold_certified_metadata
gold_certified = spark.table("metadata_governance.gold.gold_certified_metadata")
print(f"gold_certified_metadata: {gold_certified.count()} rows")
gold_certified.printSchema()
display(gold_certified.select(
    "table_name",
    "column_name",
    "certification_status",
    "error_detail",
    "ingestion_status",
    "certified_at"
).limit(10))

# Verify gold_quality_metrics
gold_metrics = spark.table("metadata_governance.gold.gold_quality_metrics")
print(f"gold_quality_metrics: {gold_metrics.count()} rows")
display(gold_metrics.orderBy("compliance_score"))

print("=" * 55)
print("  Notebook execution completed successfully")
print("=" * 55)